# Interaction-matrix diagnostics: vector and rectangular measurements

This notebook is the visual counterpart of `tests/test_interaction_matrix.py`. It uses a deterministic linear sensor so that calibration and reconstruction have an exact reference. The same six-component measurement is displayed first as a 1-D WFS vector and then as a rectangular `(2, 3)` image—no square-image assumption is required.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux.system.interaction_matrix import InteractionMatrix

torch.set_printoptions(precision=4, sci_mode=True)

## A deterministic linear DM and sensor

The sensor follows $y = A c + b$. Push-pull calibration should therefore recover $A$ exactly, independently of the static background $b$.

In [ ]:
class LinearDM:
    def __init__(self, n_commands=3, stroke=1e-6):
        self._commands = torch.nn.Parameter(torch.zeros(n_commands, dtype=torch.float64))
        self.stroke = stroke

    @property
    def commands(self):
        return self._commands

    @commands.setter
    def commands(self, value):
        value = torch.as_tensor(value, device=self._commands.device, dtype=self._commands.dtype)
        with torch.no_grad():
            self._commands.copy_(value)


dm = LinearDM()
response_matrix = torch.tensor(
    [
        [2.0, -1.0, 0.5],
        [0.5, 3.0, -0.5],
        [-4.0, 2.0, 1.0],
        [1.5, 0.25, 2.0],
        [0.0, 1.0, -1.0],
        [2.5, -0.5, 1.5],
    ],
    dtype=torch.float64,
) * 1e8
background = torch.tensor([7.0, 11.0, -5.0, 2.0, 3.0, -1.0], dtype=torch.float64)

def acquire_vector():
    return response_matrix @ dm.commands + background

In [ ]:
calibration = InteractionMatrix(
    dm=dm,
    acquiring_function=acquire_vector,
    poke_amplitude=10e-9,
)
measured_matrix = calibration.calibrate_push_pull(verbose=False)
control_matrix = calibration.compute_control_matrix(rcond=1e-12)

print(f'Matrix shape: {tuple(measured_matrix.shape)}')
print(f'Effective rank: {calibration.effective_rank}')
print(f'Retained modes: {calibration.retained_mode_indices.tolist()}')
print(f'Condition number: {calibration.condition_number:.3f}')
print(f'Max calibration error: {(measured_matrix - response_matrix).abs().max():.3e}')

## Singular values and 1-D measurement modes

In [ ]:
calibration.plot_singular_values();
calibration.plot_modes();

## The same modes as rectangular measurements

The calibration remains a flat `(n_measurements, n_commands)` matrix. `measurement_shape` affects visualization only.

In [ ]:
calibration.plot_modes(measurement_shape=(2, 3));

## Reconstruct a known command

Subtracting the reference/background response gives the differential measurement expected by the control matrix.

In [ ]:
injected = torch.tensor([80e-9, -35e-9, 20e-9], dtype=torch.float64)
differential_measurement = response_matrix @ injected
estimated = control_matrix @ differential_measurement

print('Injected commands (nm):', injected * 1e9)
print('Estimated commands (nm):', estimated * 1e9)
torch.testing.assert_close(estimated, injected, rtol=1e-10, atol=1e-15)